# 🎬 Netflix Dataset — Data Wrangling Tutorial & Practice
### `merge` · `groupby` · `fillna` · `drop_duplicates` · `astype` · PostgreSQL

**Dataset:** [Kaggle — Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**File:** `netflix_titles.csv`

| Part | Contents |
|------|----------|
| **Part 1 — Tutorial** | Each concept explained with a small runnable example |
| **Part 2 — Practice** | Apply each concept to the real Netflix dataset (💡 tips hidden) |
| **Part 3 — PostgreSQL** | How to do every operation in SQL on a Postgres database |

---


## ⚙️ Setup

Run this cell first every time you open the notebook.

In [ ]:
import pandas as pd
import json
from pathlib import Path

CSV_PATH = Path("netflix_titles.csv")   # ← point at your downloaded file

# Load and do the basic renames from the previous notebook so we start clean
df_raw = pd.read_csv(CSV_PATH, dtype=str)
df = df_raw.rename(columns={
    "show_id":      "id",
    "type":         "content_type",
    "cast":         "cast_members",
    "rating":       "maturity_rating",
    "listed_in":    "genres",
})

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


---
# Part 1 — Tutorial

Each section below explains one concept, shows the syntax, and runs a live example on a tiny dataset.  
Work top-to-bottom — each concept is self-contained.


## 1 · merge — Combining Two DataFrames

`pd.merge()` (or `df.merge()`) joins two DataFrames on a shared key column — just like SQL `JOIN`.

```python
result = pd.merge(left_df, right_df, on="key_column", how="inner")
```

### Join types

| `how=` | Keeps | SQL equivalent |
|--------|-------|----------------|
| `"inner"` | only rows where the key exists in **both** tables | `INNER JOIN` |
| `"left"` | all rows from the **left** table, `NaN` where no match on right | `LEFT JOIN` |
| `"right"` | all rows from the **right** table, `NaN` where no match on left | `RIGHT JOIN` |
| `"outer"` | all rows from **both** tables, `NaN` where no match | `FULL OUTER JOIN` |

### When keys have different names

```python
pd.merge(df_a, df_b, left_on="show_id", right_on="id")
```


In [ ]:
import io

# Two tiny tables to join
shows = pd.read_csv(io.StringIO("""id,title,type
s1,Inception,Movie
s2,The Crown,TV Show
s3,Stranger Things,TV Show
s4,The Irishman,Movie
"""))

ratings = pd.read_csv(io.StringIO("""id,imdb_score,votes
s1,8.8,2300000
s2,8.7,110000
s3,8.7,1000000
"""))  # s4 intentionally missing

print("── INNER JOIN (only matched rows) ─────────────────────────")
inner = pd.merge(shows, ratings, on="id", how="inner")
print(inner)

print("\n── LEFT JOIN (all shows, NaN where no rating) ─────────────")
left = pd.merge(shows, ratings, on="id", how="left")
print(left)


---
## 2 · groupby — Aggregating by Category

`groupby` splits the DataFrame into groups, applies a function to each group, and combines the results.

```python
df.groupby("column")["value_column"].agg_function()
```

### Common aggregation functions

| Function | What it computes |
|----------|-----------------|
| `.count()` | number of non-null rows |
| `.sum()` | total |
| `.mean()` | average |
| `.median()` | median |
| `.min()` / `.max()` | smallest / largest value |
| `.nunique()` | number of distinct values |
| `.agg({"col": "sum", "col2": "mean"})` | different functions per column |

### Reset the index after groupby

By default `groupby` makes the group key the index. Call `.reset_index()` to turn it back into a regular column:

```python
result = df.groupby("type")["title"].count().reset_index()
result.columns = ["type", "count"]
```


In [ ]:
shows_ext = pd.read_csv(io.StringIO("""id,title,type,country,year
s1,Inception,Movie,US,2010
s2,The Crown,TV Show,UK,2016
s3,Stranger Things,TV Show,US,2016
s4,The Irishman,Movie,US,2019
s5,Dark,TV Show,Germany,2017
s6,Parasite,Movie,South Korea,2019
"""))

print("── Count per type ──────────────────────────────────────────")
print(shows_ext.groupby("type")["title"].count().reset_index())

print("\n── Earliest and latest year per country ────────────────────")
print(
    shows_ext.groupby("country")["year"]
    .agg(["min", "max"])
    .reset_index()
)

print("\n── Multiple aggregations at once ───────────────────────────")
summary = shows_ext.groupby("type").agg(
    total=("id",    "count"),
    countries=("country", "nunique"),
    earliest=("year",    "min"),
).reset_index()
print(summary)


---
## 3 · fillna — Replacing Missing Values

`fillna` replaces `NaN` values with a constant, a computed value, or a strategy.

```python
df["col"].fillna("Unknown")          # constant
df["col"].fillna(df["col"].mean())   # column mean
df["col"].fillna(method="ffill")     # forward-fill from previous row
df["col"].fillna(method="bfill")     # back-fill from next row
```

### Filling multiple columns at once

```python
df.fillna({
    "director": "Unknown",
    "country":  "Unknown",
    "score":    0.0,
})
```

> **Important:** `fillna` returns a new DataFrame by default. Use `inplace=True` or reassign:  
> `df["col"] = df["col"].fillna("Unknown")`

### Checking before and after

```python
print(df.isna().sum())   # before
df = df.fillna({"director": "Unknown"})
print(df.isna().sum())   # after
```


In [ ]:
messy = pd.read_csv(io.StringIO("""id,title,director,country,score
s1,Inception,Christopher Nolan,US,8.8
s2,The Crown,,UK,8.7
s3,Stranger Things,,US,
s4,Dark,Baran bo Odar,,9.0
s5,Parasite,Bong Joon-ho,,
"""))

print("── Missing values before ───────────────────────────────────")
print(messy.isna().sum())

filled = messy.fillna({
    "director": "Unknown",
    "country":  "Unknown",
    "score":    messy["score"].mean(),   # fill with column mean
})

print("\n── Missing values after ────────────────────────────────────")
print(filled.isna().sum())
print()
print(filled)


---
## 4 · drop_duplicates — Removing Duplicate Rows

`drop_duplicates` removes rows that are identical across all (or a subset of) columns.

```python
df.drop_duplicates()                        # exact duplicate rows
df.drop_duplicates(subset=["title"])        # duplicate title only
df.drop_duplicates(subset=["title", "type"], keep="last")
```

### `keep` options

| `keep=` | Behaviour |
|---------|-----------|
| `"first"` (default) | keep the first occurrence, drop the rest |
| `"last"` | keep the last occurrence, drop the rest |
| `False` | drop **all** copies of any duplicated row |

### Spotting duplicates before dropping

```python
df[df.duplicated(subset=["title"], keep=False)]   # show all duplicate rows
df.duplicated(subset=["title"]).sum()              # count of duplicates
```


In [ ]:
dupes = pd.read_csv(io.StringIO("""id,title,type,year
s1,Inception,Movie,2010
s2,The Crown,TV Show,2016
s3,Inception,Movie,2010
s4,Inception,Movie,2012
s5,Dark,TV Show,2017
"""))

print("── All rows ────────────────────────────────────────────────")
print(dupes)

print("\n── Duplicated title rows ───────────────────────────────────")
print(dupes[dupes.duplicated(subset=["title"], keep=False)])

print("\n── keep='first' (default) ──────────────────────────────────")
print(dupes.drop_duplicates(subset=["title"], keep="first"))

print("\n── keep='last' ─────────────────────────────────────────────")
print(dupes.drop_duplicates(subset=["title"], keep="last"))

print("\n── keep=False (drop all copies) ────────────────────────────")
print(dupes.drop_duplicates(subset=["title"], keep=False))


---
## 5 · astype — Casting Column Types

`astype` converts a column to a specified dtype. Use it when you know a column is safe to convert (no bad values).  
For unsafe data, pair it with `pd.to_numeric(..., errors="coerce")` or `pd.to_datetime(..., errors="coerce")` first.

```python
df["year"]      = df["year"].astype(int)          # string → integer
df["score"]     = df["score"].astype(float)        # string → float
df["type"]      = df["type"].astype("category")    # string → Categorical
df["flag"]      = df["flag"].astype(bool)          # 0/1 → True/False
```

### Nullable integer — use when NaN may exist

```python
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int32")
#                                                              ↑ capital I
```

### Checking the result

```python
print(df.dtypes)
print(df["year"].dtype)
```

### Common dtype names

| dtype string | Pandas type |
|---|---|
| `"int64"` / `"Int64"` | integer (Int64 = nullable) |
| `"float64"` | float |
| `"str"` / `"object"` | text |
| `"category"` | Categorical |
| `"bool"` | boolean |
| `"datetime64[ns]"` | datetime (use `pd.to_datetime` instead) |


In [ ]:
raw = pd.read_csv(io.StringIO("""id,title,type,year,score,is_original
s1,Inception,Movie,2010,8.8,1
s2,The Crown,TV Show,2016,8.7,0
s3,Stranger Things,TV Show,2016,8.7,1
s4,Dark,TV Show,2017,8.8,1
"""), dtype=str)   # everything loaded as string

print("── dtypes BEFORE ───────────────────────────────────────────")
print(raw.dtypes)

typed = raw.copy()
typed["year"]        = typed["year"].astype("Int32")
typed["score"]       = typed["score"].astype(float)
typed["type"]        = typed["type"].astype("category")
typed["is_original"] = typed["is_original"].astype(bool)

print("\n── dtypes AFTER ────────────────────────────────────────────")
print(typed.dtypes)
print()
print(typed)


---
# Part 2 — Practice with Netflix Data

Apply every concept to the real dataset.  
Try each question yourself first, then click **💡 Tip** if you need the code.

> Make sure the **Setup** cell at the top has been run.


---
## Question 1 · merge

The cell below creates a small `imdb_scores` DataFrame with scores for a handful of Netflix titles.

**Task:**
1. Do a **left join** between `df` and `imdb_scores` on the `title` column.
2. Print how many rows kept their IMDB score vs. got `NaN`.
3. Do an **inner join** and explain in a comment why the row count is different.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# Left join — keeps all Netflix titles
merged_left = pd.merge(df, imdb_scores, on="title", how="left")
print("Total rows:", len(merged_left))
print("Rows with IMDB score:", merged_left["imdb_score"].notna().sum())
print("Rows without score:  ", merged_left["imdb_score"].isna().sum())

# Inner join — only titles that exist in BOTH DataFrames
merged_inner = pd.merge(df, imdb_scores, on="title", how="inner")
print("\nInner join rows:", len(merged_inner))
# Fewer rows because inner join drops any Netflix title not found in imdb_scores
```
</details>


In [ ]:
# A small IMDB scores table to join with
imdb_scores = pd.DataFrame({
    "title":      ["Inception", "The Crown", "Stranger Things", "Dark", "Parasite",
                   "Breaking Bad", "The Witcher", "Money Heist"],
    "imdb_score": [8.8, 8.7, 8.7, 8.8, 8.6, 9.5, 8.2, 8.3],
})

# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 2 · groupby

Using the Netflix DataFrame `df`:

1. Count how many titles exist per `content_type` (Movie vs TV Show).
2. Find the **top 5 countries** by number of titles. Handle the fact that `country` may be missing.
3. For each `content_type`, find the **most common maturity rating** (hint: `value_counts` inside `agg` or use `groupby` + `transform`).

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# 1. Count per content type
counts = df.groupby("content_type")["id"].count().reset_index()
counts.columns = ["content_type", "total"]
print(counts)

# 2. Top 5 countries (drop missing first)
top_countries = (
    df.dropna(subset=["country"])
    .groupby("country")["id"]
    .count()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
top_countries.columns = ["country", "total"]
print(top_countries)

# 3. Most common rating per content type
most_common_rating = (
    df.groupby("content_type")["maturity_rating"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
most_common_rating.columns = ["content_type", "most_common_rating"]
print(most_common_rating)
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 3 · fillna

1. Print the missing value count for every column in `df`.
2. Fill the missing values using the table below.
3. Verify that all target columns now show 0 missing values.

| Column | Fill value |
|--------|-----------|
| `director` | `"Unknown"` |
| `cast_members` | `"Unknown"` |
| `country` | `"Unknown"` |
| `date_added` | `"Unknown"` |
| `maturity_rating` | `"NR"` |
| `duration` | `"Unknown"` |

**Bonus:** For `release_year`, fill any missing with the **median** release year (convert to numeric first).

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# 1. Missing before
print("Before:\n", df.isna().sum())

# 2. Fill
fill_map = {
    "director":        "Unknown",
    "cast_members":    "Unknown",
    "country":         "Unknown",
    "date_added":      "Unknown",
    "maturity_rating": "NR",
    "duration":        "Unknown",
}
df = df.fillna(fill_map)

# Bonus: fill release_year with median
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
median_year = int(df["release_year"].median())
df["release_year"] = df["release_year"].fillna(median_year)

# 3. Verify
print("\nAfter:\n", df[list(fill_map.keys()) + ["release_year"]].isna().sum())
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 4 · drop_duplicates

1. Check if there are any rows in `df` with the **same `title` and `content_type`** combination.
2. Print those duplicated rows.
3. Drop duplicates keeping the **first** occurrence and confirm the row count before and after.
4. **Bonus:** Drop rows where the entire row is a duplicate (all 12 columns identical).

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# 1. Count duplicates
dupe_count = df.duplicated(subset=["title", "content_type"]).sum()
print(f"Duplicate (title + content_type) rows: {dupe_count}")

# 2. Show them
dupes = df[df.duplicated(subset=["title", "content_type"], keep=False)]
print(dupes[["id", "title", "content_type", "release_year"]].sort_values("title"))

# 3. Drop, keep first
print("\nRows before:", len(df))
df = df.drop_duplicates(subset=["title", "content_type"], keep="first")
print("Rows after: ", len(df))

# Bonus: exact full-row duplicates
exact_dupes = df.duplicated().sum()
print(f"\nExact full-row duplicates: {exact_dupes}")
df = df.drop_duplicates()
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 5 · astype

Cast the following columns to the right types and print `df.dtypes` to confirm.

| Column | Target type | Notes |
|--------|-------------|-------|
| `release_year` | nullable `Int32` | use `pd.to_numeric(..., errors="coerce")` first |
| `date_added` | `datetime64` | format `"%B %d, %Y"`, coerce bad values |
| `content_type` | `category` | only 2 unique values |
| `maturity_rating` | `category` | limited set of ratings |

**Bonus:** Extract just the **year** from `date_added` as a new integer column called `year_added`.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# release_year → nullable Int32
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int32")

# date_added → datetime
df["date_added"] = pd.to_datetime(
    df["date_added"].str.strip().replace("Unknown", pd.NaT),
    format="%B %d, %Y",
    errors="coerce",
)

# Categorical columns
df["content_type"]    = df["content_type"].astype("category")
df["maturity_rating"] = df["maturity_rating"].astype("category")

print(df.dtypes)

# Bonus: extract year added
df["year_added"] = df["date_added"].dt.year.astype("Int32")
print(df[["title", "date_added", "year_added"]].head())
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 6 · Putting It All Together

Build a clean summary table using all five concepts:

1. **fillna** — fill `country` and `director` with `"Unknown"` if not done already
2. **drop_duplicates** — remove any duplicate `title` + `content_type` rows
3. **astype** — cast `release_year` to `Int32` and `content_type` to `category`
4. **groupby** — count titles per country, keep only countries with **≥ 10 titles**
5. **merge** — join that country summary back onto the main DataFrame so each row has a `country_total` column

Print the first 10 rows of the final result sorted by `country_total` descending.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# 1. fillna
df["country"]  = df["country"].fillna("Unknown")
df["director"] = df["director"].fillna("Unknown")

# 2. drop_duplicates
df = df.drop_duplicates(subset=["title", "content_type"], keep="first")

# 3. astype
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int32")
df["content_type"] = df["content_type"].astype("category")

# 4. groupby → country totals with ≥ 10 titles
country_counts = (
    df.groupby("country")["id"]
    .count()
    .reset_index()
    .rename(columns={"id": "country_total"})
)
country_counts = country_counts[country_counts["country_total"] >= 10]

# 5. merge back
df_final = pd.merge(df, country_counts, on="country", how="inner")

print(df_final[["title", "content_type", "country", "country_total", "release_year"]]
      .sort_values("country_total", ascending=False)
      .head(10))
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
# Part 3 — The Same Logic in PostgreSQL

Every pandas operation above has a direct SQL equivalent.  
This section shows the mapping side-by-side so you can apply the same logic in a Postgres database.

Assume the Netflix data lives in a table called **`netflix_titles`** with these columns:

```
id, content_type, title, director, cast_members, country,
date_added, release_year, maturity_rating, duration, genres, description
```

And a second table **`imdb_scores`** with columns `title` and `imdb_score`.


---
## 3.1 · merge → JOIN

### pandas
```python
# Left join
pd.merge(df, imdb_scores, on="title", how="left")

# Inner join
pd.merge(df, imdb_scores, on="title", how="inner")
```

### PostgreSQL
```sql
-- Left join: all Netflix titles, NULLs where no IMDB score
SELECT n.*, i.imdb_score
FROM   netflix_titles n
LEFT JOIN imdb_scores i ON n.title = i.title;

-- Inner join: only titles that appear in both tables
SELECT n.*, i.imdb_score
FROM   netflix_titles n
INNER JOIN imdb_scores i ON n.title = i.title;

-- Full outer join (pandas how="outer")
SELECT n.*, i.imdb_score
FROM   netflix_titles n
FULL OUTER JOIN imdb_scores i ON n.title = i.title;
```

### Key differences
- SQL `NULL` = pandas `NaN` for unmatched rows in outer joins.
- In pandas you can join on multiple columns: `on=["col_a", "col_b"]`. In SQL: `ON a.col_a = b.col_a AND a.col_b = b.col_b`.
- Column name conflicts are handled with aliases in SQL (`SELECT a.id AS netflix_id`) vs. suffixes in pandas (`suffixes=("_left", "_right")`).


---
## 3.2 · groupby → GROUP BY

### pandas
```python
# Count per content type
df.groupby("content_type")["id"].count().reset_index()

# Multiple aggregations
df.groupby("content_type").agg(
    total     = ("id",      "count"),
    countries = ("country", "nunique"),
    earliest  = ("release_year", "min"),
).reset_index()

# Filter groups (only those with ≥ 10 titles)
df.groupby("country").filter(lambda x: len(x) >= 10)
```

### PostgreSQL
```sql
-- Count per content type
SELECT content_type, COUNT(*) AS total
FROM   netflix_titles
GROUP BY content_type;

-- Multiple aggregations
SELECT
    content_type,
    COUNT(*)                     AS total,
    COUNT(DISTINCT country)      AS countries,
    MIN(release_year)            AS earliest
FROM   netflix_titles
GROUP BY content_type;

-- Filter groups: HAVING is the SQL equivalent of pandas .filter()
SELECT country, COUNT(*) AS total
FROM   netflix_titles
GROUP BY country
HAVING COUNT(*) >= 10
ORDER BY total DESC;
```

### Key differences
- `HAVING` filters **after** grouping; `WHERE` filters **before** grouping.
- `COUNT(*)` counts all rows; `COUNT(column)` skips NULLs — same as pandas `.count()`.
- `COUNT(DISTINCT column)` = pandas `.nunique()`.


---
## 3.3 · fillna → COALESCE / UPDATE

### pandas
```python
# Fill a single column
df["director"] = df["director"].fillna("Unknown")

# Fill multiple columns at once
df = df.fillna({"director": "Unknown", "country": "Unknown"})

# Fill with column mean
df["score"] = df["score"].fillna(df["score"].mean())
```

### PostgreSQL

**In a query (non-destructive):**
```sql
-- COALESCE returns the first non-NULL argument
SELECT
    id,
    title,
    COALESCE(director, 'Unknown')  AS director,
    COALESCE(country,  'Unknown')  AS country
FROM netflix_titles;
```

**Persistent update (modifies the table):**
```sql
-- Fill NULLs permanently in the table
UPDATE netflix_titles
SET    director = 'Unknown'
WHERE  director IS NULL;

UPDATE netflix_titles
SET    country = 'Unknown'
WHERE  country IS NULL;
```

**Fill with a computed value (e.g. average):**
```sql
UPDATE netflix_titles
SET    release_year = (SELECT ROUND(AVG(release_year)) FROM netflix_titles WHERE release_year IS NOT NULL)
WHERE  release_year IS NULL;
```

### Key differences
- `COALESCE` is read-only; `UPDATE ... SET ... WHERE ... IS NULL` mutates the table.
- SQL `NULL` is contagious in arithmetic just like pandas `NaN` — `NULL + 5 = NULL`.
- Use `NULLIF(column, 'Unknown')` to go the other direction (replace a sentinel back with NULL).


---
## 3.4 · drop_duplicates → DISTINCT / ROW_NUMBER

### pandas
```python
# Remove exact full-row duplicates
df.drop_duplicates()

# Remove duplicates on a subset of columns, keep first
df.drop_duplicates(subset=["title", "content_type"], keep="first")

# Show duplicated rows
df[df.duplicated(subset=["title"], keep=False)]
```

### PostgreSQL

**Simple deduplication (SELECT DISTINCT):**
```sql
-- Unique content types
SELECT DISTINCT content_type FROM netflix_titles;

-- Unique title + content_type combinations
SELECT DISTINCT title, content_type FROM netflix_titles;
```

**Keep one row per duplicate group — equivalent to keep="first":**
```sql
-- ROW_NUMBER assigns 1 to the "first" row in each group
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY title, content_type   -- group by these columns
               ORDER BY id                        -- "first" = lowest id
           ) AS rn
    FROM netflix_titles
)
SELECT * FROM ranked WHERE rn = 1;
```

**Find duplicated rows:**
```sql
SELECT title, content_type, COUNT(*) AS copies
FROM   netflix_titles
GROUP  BY title, content_type
HAVING COUNT(*) > 1
ORDER  BY copies DESC;
```

**Delete duplicates from the table:**
```sql
DELETE FROM netflix_titles
WHERE id NOT IN (
    SELECT MIN(id)
    FROM   netflix_titles
    GROUP BY title, content_type
);
```

### Key differences
- `SELECT DISTINCT` only returns distinct *projected* columns, not full row dedup — use the `ROW_NUMBER()` approach when you need all columns.
- `PARTITION BY` in SQL ≈ `groupby` key in pandas.
- `ORDER BY` inside `OVER(...)` determines which row is "first" — pandas uses the physical row order.


---
## 3.5 · astype → CAST / ALTER COLUMN

### pandas
```python
df["release_year"] = df["release_year"].astype("Int32")
df["content_type"] = df["content_type"].astype("category")
df["date_added"]   = pd.to_datetime(df["date_added"], errors="coerce")
```

### PostgreSQL

**In a query (cast without changing the table):**
```sql
SELECT
    id,
    title,
    CAST(release_year AS INTEGER)       AS release_year,
    CAST(date_added   AS DATE)          AS date_added,
    date_added::TIMESTAMP               -- shorthand :: syntax
FROM netflix_titles;
```

**Parse a formatted date string:**
```sql
SELECT TO_DATE(date_added, 'Month DD, YYYY') AS date_added
FROM   netflix_titles
WHERE  date_added IS NOT NULL;
```

**Change the column type permanently:**
```sql
-- Change release_year from TEXT to INTEGER
ALTER TABLE netflix_titles
ALTER COLUMN release_year TYPE INTEGER
USING release_year::INTEGER;

-- Change date_added from TEXT to DATE with format parsing
ALTER TABLE netflix_titles
ALTER COLUMN date_added TYPE DATE
USING TO_DATE(date_added, 'Month DD, YYYY');
```

### Common type mappings

| pandas dtype | PostgreSQL type |
|---|---|
| `object` / `str` | `TEXT` or `VARCHAR` |
| `Int32` / `Int64` | `INTEGER` / `BIGINT` |
| `float64` | `DOUBLE PRECISION` or `NUMERIC` |
| `datetime64[ns]` | `TIMESTAMP` or `DATE` |
| `bool` | `BOOLEAN` |
| `category` | `TEXT` + a `CHECK` constraint or a lookup table |

### Key differences
- Postgres `CAST` raises an error on bad values; pandas `errors="coerce"` silently produces `NaN`.  
  Use `NULLIF` + a `CASE` expression to replicate coerce behaviour:
  ```sql
  CAST(NULLIF(release_year, '') AS INTEGER)
  ```
- There is no Categorical type in SQL — model it with a lookup/reference table and a foreign key.
- `ALTER COLUMN` is a schema-changing DDL statement; it requires appropriate table privileges.


---
## 3.6 · Quick Reference Cheat-Sheet

| Operation | pandas | PostgreSQL |
|-----------|--------|-----------|
| Join tables | `pd.merge(a, b, on="key", how="left")` | `SELECT ... FROM a LEFT JOIN b ON a.key = b.key` |
| Aggregate | `df.groupby("col").agg(n=("id","count"))` | `SELECT col, COUNT(*) AS n FROM t GROUP BY col` |
| Filter groups | `.groupby().filter(lambda x: len(x) >= 10)` | `GROUP BY col HAVING COUNT(*) >= 10` |
| Fill missing | `df["col"].fillna("Unknown")` | `COALESCE(col, 'Unknown')` or `UPDATE ... WHERE col IS NULL` |
| Check missing | `df["col"].isna().sum()` | `SELECT COUNT(*) FROM t WHERE col IS NULL` |
| Remove dupes | `df.drop_duplicates(subset=["a","b"])` | `SELECT DISTINCT a, b ...` or `ROW_NUMBER() OVER (PARTITION BY ...)` |
| Keep first dupe | `keep="first"` | `ROW_NUMBER() ... ORDER BY id` → `WHERE rn = 1` |
| Cast type | `df["col"].astype("Int32")` | `CAST(col AS INTEGER)` or `col::INTEGER` |
| Safe cast | `pd.to_numeric(..., errors="coerce")` | `CAST(NULLIF(col, '') AS INTEGER)` |
| Change schema | `df["col"].astype(...)` (in memory) | `ALTER TABLE t ALTER COLUMN col TYPE ...` |


---
## 🏆 Bonus Challenges

**Bonus 1 — Merge + groupby pipeline**  
Join `df` with a DataFrame of your own IMDB scores for 20+ titles, then group by `content_type` to find the average IMDB score for Movies vs TV Shows.

**Bonus 2 — Nested groupby**  
Find the top director (by title count) for each `content_type`. Hint: `groupby(["content_type", "director"])` then use `idxmax()`.

**Bonus 3 — SQL window function**  
Write a SQL query using `ROW_NUMBER() OVER (PARTITION BY country ORDER BY release_year DESC)` to find the most recent title added per country.

**Bonus 4 — Round-trip**  
Write the deduplicated, type-cast DataFrame to Parquet, reload it, and then verify the schema with `df.dtypes` matches what you expect.


In [ ]:
# ── Bonus workspace ───────────────────────────────────────────────────────



